## 🎯 Learning Objectives
* Understand the concept of overfitting in deep learning models.
* Implement and apply various regularisation techniques (e.g., L1/L2 regularisation, Dropout, Early Stopping) using PyTorch.
* Evaluate the effectiveness of regularisation techniques by comparing model performance on training and validation sets.
* Select appropriate regularisation strategies for a given deep learning task.


# DL01-L12: Exercise: Improve a Model with Regularisation Techniques

## Task: Combat Overfitting in a Neural Network

**Context:** As an ML engineer transitioning into deep learning, you've built your first neural networks. You've noticed that while your models perform exceptionally well on training data, their performance often degrades significantly on unseen validation data – a classic sign of overfitting. This exercise challenges you to apply various regularisation techniques to mitigate this issue and build a more robust model.

**Objective:** You will be provided with a baseline PyTorch neural network that is designed to overfit a synthetic dataset. Your task is to modify this baseline model and its training process by incorporating at least two different regularisation techniques to improve its generalisation performance.

### Dataset
You will work with a synthetic dataset generated using `sklearn.datasets.make_moons`, which presents a non-linearly separable binary classification problem. This dataset is intentionally small to make overfitting more apparent.

### Requirements:
1.  **Baseline Model Analysis:** Run the provided baseline model and observe its training and validation loss/accuracy curves. Clearly identify the signs of overfitting.
2.  **Implement Regularisation:** Choose at least two of the following regularisation techniques and integrate them into the model architecture or training loop:
    *   **L1/L2 Regularisation (Weight Decay):** Apply L1 or L2 regularisation to the model's weights.
    *   **Dropout:** Add Dropout layers to your neural network.
    *   **Batch Normalisation:** Incorporate Batch Normalisation layers.
    *   **Early Stopping:** Implement an early stopping mechanism in your training loop.
    *   *(Optional but encouraged)* Data Augmentation: If you feel adventurous, consider simple data augmentation techniques if applicable to the synthetic data (e.g., adding noise).
3.  **Train and Evaluate:** Train your regularised model using the same training and validation splits as the baseline.
4.  **Compare Performance:** Plot the training and validation loss/accuracy curves for both the baseline and your regularised model side-by-side. Clearly articulate how regularisation has impacted the model's performance and generalisation capabilities.
5.  **Code Clarity:** Ensure your code is well-commented, explaining your choices for regularisation techniques and their parameters.

### Evaluation Criteria:
*   **Correctness:** Proper implementation of chosen regularisation techniques.
*   **Effectiveness:** Demonstrable improvement in validation performance compared to the baseline.
*   **Analysis:** Clear explanation of overfitting in the baseline and how regularisation addressed it.
*   **Code Quality:** Readability, comments, and adherence to PyTorch best practices.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

# --- Configuration --- #
RANDOM_SEED = 42
BATCH_SIZE = 32
LEARNING_RATE = 0.01
NUM_EPOCHS = 200 # High epochs to ensure overfitting for baseline

# Set random seeds for reproducibility
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# --- 1. Generate Synthetic Dataset --- #
def generate_data(n_samples=1000, noise=0.2, random_state=RANDOM_SEED):
    X, y = make_moons(n_samples=n_samples, noise=noise, random_state=random_state)
    # Scale features
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32).unsqueeze(1) # Binary classification, unsqueeze for BCEWithLogitsLoss
    return X, y

X, y = generate_data()

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

# Create DataLoader objects
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

# --- 2. Define Baseline Model (Designed to Overfit) --- #
class BaselineModel(nn.Module):
    def __init__(self):
        super(BaselineModel, self).__init__()
        self.layer1 = nn.Linear(2, 128) # Input features: 2
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(128, 128)
        self.relu2 = nn.ReLU()
        self.layer3 = nn.Linear(128, 64)
        self.relu3 = nn.ReLU()
        self.output_layer = nn.Linear(64, 1) # Output features: 1 for binary classification

    def forward(self, x):
        x = self.relu1(self.layer1(x))
        x = self.relu2(self.layer2(x))
        x = self.relu3(self.layer3(x))
        x = self.output_layer(x)
        return x

# --- 3. Training Function --- #
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, model_name="Model"):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    print(f"\n--- Training {model_name} on {device} ---")

    for epoch in range(num_epochs):
        model.train() # Set model to training mode
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = correct_train / total_train
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)

        # Validation phase
        model.eval() # Set model to evaluation mode
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        with torch.no_grad(): # Disable gradient calculation for validation
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                predicted = (torch.sigmoid(outputs) > 0.5).float()
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = correct_val / total_val
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)

        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], ' \
                  f'Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f}, ' \
                  f'Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}')

    print(f"--- {model_name} Training Complete ---")
    return history

# --- 4. Plotting Function --- #
def plot_history(baseline_history, regularized_history, num_epochs):
    epochs = range(1, num_epochs + 1)

    plt.figure(figsize=(14, 6))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, baseline_history['train_loss'], 'r--', label='Baseline Train Loss')
    plt.plot(epochs, baseline_history['val_loss'], 'r-', label='Baseline Val Loss')
    plt.plot(epochs, regularized_history['train_loss'], 'b--', label='Regularized Train Loss')
    plt.plot(epochs, regularized_history['val_loss'], 'b-', label='Regularized Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, baseline_history['train_acc'], 'r--', label='Baseline Train Accuracy')
    plt.plot(epochs, baseline_history['val_acc'], 'r-', label='Baseline Val Accuracy')
    plt.plot(epochs, regularized_history['train_acc'], 'b--', label='Regularized Train Accuracy')
    plt.plot(epochs, regularized_history['val_acc'], 'b-', label='Regularized Val Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# --- 5. Train Baseline Model --- #
print("\n--- Training Baseline Model ---")
baseline_model = BaselineModel()
baseline_criterion = nn.BCEWithLogitsLoss()
baseline_optimizer = optim.Adam(baseline_model.parameters(), lr=LEARNING_RATE)

baseline_history = train_model(baseline_model, train_loader, val_loader, baseline_criterion, baseline_optimizer, NUM_EPOCHS, "Baseline Model")

print("\nBaseline Model Training Complete. Observe the gap between training and validation metrics.")


## Your Turn: Implement Regularisation!

Now it's your turn to improve the model. Based on the baseline's performance, you should see clear signs of overfitting (training accuracy much higher than validation accuracy, validation loss increasing while training loss decreases).

**Your task is to:**

1.  **Define a new model class** (e.g., `RegularizedModel`) that extends `nn.Module`.
2.  **Incorporate at least two regularisation techniques** into this new model or its training process. Good candidates include:
    *   **Dropout layers** (`nn.Dropout`) after activation functions.
    *   **L2 Regularisation (Weight Decay)** by passing `weight_decay` to your optimizer (e.g., `optim.Adam(..., weight_decay=0.001)`).
    *   **Batch Normalisation layers** (`nn.BatchNorm1d`) before activation functions.
    *   **Early Stopping** logic within the training loop (you might need to modify the `train_model` function or create a new one).
3.  **Instantiate and train** your `RegularizedModel`.
4.  **Compare the results** using the `plot_history` function provided. Analyze how your chosen regularisation techniques have helped reduce overfitting and improve generalisation.

Think about where to place Dropout layers, appropriate `p` values, and suitable `weight_decay` values. Experimentation is key!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

# --- Configuration (re-defined for solution clarity) --- #
RANDOM_SEED = 42
BATCH_SIZE = 32
LEARNING_RATE = 0.01
NUM_EPOCHS = 200 # High epochs to ensure overfitting for baseline, but early stopping will manage this for regularized model

# Set random seeds for reproducibility
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# --- 1. Generate Synthetic Dataset (re-run for self-contained solution) --- #
def generate_data(n_samples=1000, noise=0.2, random_state=RANDOM_SEED):
    X, y = make_moons(n_samples=n_samples, noise=noise, random_state=random_state)
    # Scale features
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32).unsqueeze(1) # Binary classification, unsqueeze for BCEWithLogitsLoss
    return X, y

X, y = generate_data()

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

# Create DataLoader objects
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# --- 2. Define Baseline Model (re-run for self-contained solution) --- #
class BaselineModel(nn.Module):
    def __init__(self):
        super(BaselineModel, self).__init__()
        self.layer1 = nn.Linear(2, 128)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(128, 128)
        self.relu2 = nn.ReLU()
        self.layer3 = nn.Linear(128, 64)
        self.relu3 = nn.ReLU()
        self.output_layer = nn.Linear(64, 1)

    def forward(self, x):
        x = self.relu1(self.layer1(x))
        x = self.relu2(self.layer2(x))
        x = self.relu3(self.layer3(x))
        x = self.output_layer(x)
        return x

# --- 3. Training Function (Modified for Early Stopping) --- #
def train_model_with_early_stopping(model, train_loader, val_loader, criterion, optimizer, num_epochs, model_name="Model", patience=10, min_delta=0.001):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    print(f"\n--- Training {model_name} on {device} ---")

    best_val_loss = float('inf')
    epochs_no_improve = 0
    early_stop = False

    for epoch in range(num_epochs):
        model.train() # Set model to training mode
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = correct_train / total_train
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)

        # Validation phase
        model.eval() # Set model to evaluation mode
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        with torch.no_grad(): # Disable gradient calculation for validation
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                predicted = (torch.sigmoid(outputs) > 0.5).float()
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = correct_val / total_val
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)

        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], ' \
                  f'Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f}, ' \
                  f'Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}')

        # Early Stopping check
        if epoch_val_loss < best_val_loss - min_delta:
            best_val_loss = epoch_val_loss
            epochs_no_improve = 0
            # Optionally save the best model state
            # torch.save(model.state_dict(), f'{model_name}_best_model.pth')
        else:
            epochs_no_improve += 1
            if epochs_no_improve == patience:
                print(f'Early stopping triggered after {epoch+1} epochs (no improvement for {patience} epochs).')
                early_stop = True
                break

    print(f"--- {model_name} Training Complete ---")
    return history

# --- 4. Plotting Function (re-run for self-contained solution) --- #
def plot_history(baseline_history, regularized_history, num_epochs_baseline, num_epochs_regularized):
    # Adjust epochs for plotting based on actual training length
    epochs_baseline = range(1, num_epochs_baseline + 1)
    epochs_regularized = range(1, num_epochs_regularized + 1)

    plt.figure(figsize=(14, 6))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs_baseline, baseline_history['train_loss'], 'r--', label='Baseline Train Loss')
    plt.plot(epochs_baseline, baseline_history['val_loss'], 'r-', label='Baseline Val Loss')
    plt.plot(epochs_regularized, regularized_history['train_loss'], 'b--', label='Regularized Train Loss')
    plt.plot(epochs_regularized, regularized_history['val_loss'], 'b-', label='Regularized Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs_baseline, baseline_history['train_acc'], 'r--', label='Baseline Train Accuracy')
    plt.plot(epochs_baseline, baseline_history['val_acc'], 'r-', label='Baseline Val Accuracy')
    plt.plot(epochs_regularized, regularized_history['train_acc'], 'b--', label='Regularized Train Accuracy')
    plt.plot(epochs_regularized, regularized_history['val_acc'], 'b-', label='Regularized Val Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# --- 5. Train Baseline Model (re-run for self-contained solution) --- #
print("\n--- Training Baseline Model ---")
baseline_model = BaselineModel()
baseline_criterion = nn.BCEWithLogitsLoss()
baseline_optimizer = optim.Adam(baseline_model.parameters(), lr=LEARNING_RATE)

baseline_history = train_model_with_early_stopping(baseline_model, train_loader, val_loader, baseline_criterion, baseline_optimizer, NUM_EPOCHS, "Baseline Model", patience=NUM_EPOCHS+1, min_delta=0) # Effectively no early stopping for baseline

print("\nBaseline Model Training Complete. Observe the gap between training and validation metrics.")

# --- Reference Solution: Regularized Model Implementation --- #

class RegularizedModel(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super(RegularizedModel, self).__init__()
        # Layer 1: Linear -> BatchNorm -> ReLU -> Dropout
        self.layer1 = nn.Linear(2, 128)
        self.bn1 = nn.BatchNorm1d(128) # Batch Normalization after linear transformation
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate) # Dropout to randomly set activations to zero

        # Layer 2: Linear -> BatchNorm -> ReLU -> Dropout
        self.layer2 = nn.Linear(128, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)

        # Layer 3: Linear -> BatchNorm -> ReLU -> Dropout
        self.layer3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout_rate)

        # Output Layer
        self.output_layer = nn.Linear(64, 1)

    def forward(self, x):
        x = self.dropout1(self.relu1(self.bn1(self.layer1(x))))
        x = self.dropout2(self.relu2(self.bn2(self.layer2(x))))
        x = self.dropout3(self.relu3(self.bn3(self.layer3(x))))
        x = self.output_layer(x)
        return x

# --- Instantiate and Train Regularized Model --- #
print("\n--- Training Regularized Model ---")

# Regularization techniques applied:
# 1. Dropout: Added nn.Dropout layers in the model definition.
# 2. L2 Regularization (Weight Decay): Applied via the optimizer.
# 3. Batch Normalization: Added nn.BatchNorm1d layers in the model definition.
# 4. Early Stopping: Implemented in the modified train_model_with_early_stopping function.

regularized_model = RegularizedModel(dropout_rate=0.4) # Experiment with dropout rate (e.g., 0.2 to 0.5)
regularized_criterion = nn.BCEWithLogitsLoss()

# Apply L2 regularization (weight_decay) to the optimizer
# A common value for weight_decay is 0.0001 to 0.01
regularized_optimizer = optim.Adam(regularized_model.parameters(), lr=LEARNING_RATE, weight_decay=0.001)

# Train with Early Stopping
# Patience: Number of epochs with no improvement after which training will be stopped.
# min_delta: Minimum change in the monitored quantity to qualify as an improvement.
regularized_history = train_model_with_early_stopping(
    regularized_model, train_loader, val_loader, regularized_criterion, 
    regularized_optimizer, NUM_EPOCHS, "Regularized Model", 
    patience=20, min_delta=0.0005 # Adjust patience and min_delta as needed
)

# --- Compare Performance --- #
print("\n--- Comparing Baseline vs. Regularized Model Performance ---")
plot_history(baseline_history, regularized_history, len(baseline_history['train_loss']), len(regularized_history['train_loss']))

print("\nAnalysis:")
print("The baseline model shows a significant gap between training and validation accuracy/loss, indicating overfitting. Training loss continues to decrease, while validation loss starts to increase or plateau early.")
print("The regularized model, incorporating Dropout, Batch Normalization, L2 regularization (weight decay), and Early Stopping, exhibits a much smaller gap between training and validation metrics. Its validation loss tends to be lower and more stable, and validation accuracy is higher, demonstrating improved generalization. Early stopping also prevents unnecessary training once validation performance plateaus.")
print("This exercise highlights how crucial regularisation techniques are for building robust deep learning models that perform well on unseen data, moving beyond simply memorizing the training set.")
